# Step 6 — SageMaker Pipelines (CI/CD)

Automate the full workflow as a DAG with two steps.

In [ ]:
import boto3
import sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.huggingface import HuggingFace
from sagemaker.inputs import TrainingInput

BUCKET = "YOUR-S3-BUCKET-NAME"
REGION = "us-east-2"
PREFIX = "symptom-data"

boto_session      = boto3.Session(region_name=REGION)
pipeline_session  = PipelineSession(
    boto_session=boto_session,
    default_bucket=BUCKET
)
role = sagemaker.get_execution_role()

In [ ]:
# Step 1: Preprocessing
processor = ScriptProcessor(
    image_uri="763104351884.dkr.ecr.us-east-2.amazonaws.com/pytorch-training:2.0.0-cpu-py310",
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    command=["python3"],
    sagemaker_session=pipeline_session,
)
step_process = ProcessingStep(
    name="PreprocessData",
    processor=processor,
    code="../scripts/preprocessing.py",
    inputs=[ProcessingInput(
        source=f"s3://{BUCKET}/{PREFIX}/raw/",
        destination="/opt/ml/processing/input"
    )],
    outputs=[ProcessingOutput(
        source="/opt/ml/processing/output",
        destination=f"s3://{BUCKET}/{PREFIX}/processed/"
    )]
)

# Step 2: Training (depends on Step 1)
estimator = HuggingFace(
    entry_point="train.py",
    source_dir="../scripts",
    role=role,
    instance_type="ml.g4dn.xlarge",
    instance_count=1,
    transformers_version="4.26",
    pytorch_version="1.13",
    py_version="py39",
    hyperparameters={"epochs": 3, "batch-size": 16},
    sagemaker_session=pipeline_session,
)
step_train = TrainingStep(
    name="TrainBioBERT",
    estimator=estimator,
    inputs={"train": TrainingInput(s3_data=f"s3://{BUCKET}/{PREFIX}/processed/")}
)

# Build and run pipeline
pipeline = Pipeline(
    name="SymptomClassificationPipeline",
    steps=[step_process, step_train],
    sagemaker_session=pipeline_session
)
pipeline.upsert(role_arn=role)
execution = pipeline.start()
print(f"Pipeline started: {execution.arn}")
print("Check AWS Console → SageMaker → Pipelines for the DAG view")

## Check execution status

In [ ]:
import boto3
sm_client = boto3.client("sagemaker", region_name="us-east-2")

executions = sm_client.list_pipeline_executions(
    PipelineName="SymptomClassificationPipeline"
)
for exe in executions["PipelineExecutionSummaries"]:
    print(f"Status: {exe['PipelineExecutionStatus']}")
    print(f"ARN:    {exe['PipelineExecutionArn']}")